# 01.02 OpenCV 图像处理基础

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 01.01 章节概述</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">理解图像即数字矩阵，掌握 OpenCV 读写、灰度转换、绘图操作，理解 BGR/RGB 差异</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">图像的本质 → OpenCV 基础操作 → BGR/RGB → 绘图函数</td></tr>
</table>

## 第一部分：图像的本质——图片就是数字矩阵


In [ ]:
# 安装依赖（清华源加速）
!pip install numpy opencv-python matplotlib -i https://pypi.tuna.tsinghua.edu.cn/simple -q

import numpy as np
import cv2
import matplotlib.pyplot as plt

# 中文字体配置（防止中文标题乱码）
from matplotlib import font_manager
for f in ['Arial Unicode MS', 'Heiti SC', 'PingFang SC', 'Microsoft YaHei', 'SimHei']:
    if f in {x.name for x in font_manager.fontManager.ttflist}:
        plt.rcParams['font.sans-serif'] = [f]
        break
plt.rcParams['axes.unicode_minus'] = False

print("✅ OpenCV 版本:", cv2.__version__)
print("✅ NumPy 版本:", np.__version__)


## 1. 灰度图：一张图就是一个二维数组

在计算机眼中，**一张灰度图就是一个二维数字表格（矩阵）**：

- 表格的每个格子叫一个**像素（pixel）**；
- 每个像素是一个 **0 到 255 的整数**：`0` = 纯黑，`255` = 纯白，中间值 = 不同深浅的灰色。

我们用 NumPy 手动构造一张 100×100 的渐变图（从左到右由黑变白），直观感受这一点：

In [ ]:
# 用 NumPy 手动构造一张灰度图（不依赖任何外部文件）
img = np.zeros((100, 100), dtype=np.uint8)   # 先创建 100x100 全黑图（全 0）
# dtype=np.uint8：每个像素用 8 位无符号整数表示（范围 0-255）

for x in range(100):
    img[:, x] = int(x * 2.55)    # 第 x 列全部赋值为 x*2.55（从 0 渐变到 255）

print("图像形状:", img.shape, "（高100 × 宽100）")
print("左上角像素值:", img[0, 0], "（纯黑）")
print("右上角像素值:", img[0, -1], "（接近纯白）")

plt.imshow(img, cmap='gray')   # cmap='gray' 表示用灰度色图显示
plt.title("Hand-crafted Grayscale Gradient")
plt.axis('off')
plt.show()


> 💡 看！一堆 0-255 的数字，显示出来就是一张图。这就是"图像 = 数字矩阵"的真相。

### 理解 `shape`

`img.shape` 返回 `(高度, 宽度)`。注意 OpenCV/NumPy 中**第一个维度是高度（行），第二个是宽度（列）**，这和直觉相反，是初学者常踩的坑。

## 2. 彩色图：三个通道的"三明治"

彩色图比灰度图多了一个维度——**通道（channel）**。最常见的彩色图用 **RGB 三通道**：R（红）、G（绿）、B（蓝），三种颜色按不同比例混合就能表示几乎所有颜色。所以一张彩色图是 **三维数组**：`(高度, 宽度, 3)`。

> ⚠️ **重要提醒**：OpenCV 默认使用 **BGR** 顺序（蓝-绿-红），而 Matplotlib 使用 **RGB** 顺序。这个差异会导致颜色"反掉"——我们会在 01.03 节详细处理。本节先用 RGB 顺序演示。

我们用 NumPy 构造一张左红右黄的彩色图：

In [ ]:
# 用 NumPy 构造一张 100x100 的彩色图（3 通道）
img_color = np.zeros((100, 100, 3), dtype=np.uint8)   # 形状 (100, 100, 3)

img_color[:, :50] = [255, 0, 0]      # 左半边涂红色（R=255, G=0, B=0）
img_color[:, 50:] = [255, 255, 0]    # 右半边涂黄色（R=255, G=255, B=0）

print("彩色图形状:", img_color.shape, "（高 × 宽 × 3通道）")
print("左半边一个像素的值:", img_color[50, 25], "= 红")
print("右半边一个像素的值:", img_color[50, 75], "= 黄")

plt.imshow(img_color)   # Matplotlib 默认按 RGB 显示
plt.title("Color Image = 3 Layers Stacked")
plt.axis('off')
plt.show()


### 常见颜色的 RGB 值

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">颜色</th><th align="left">R</th><th align="left">G</th><th align="left">B</th></tr>
<tr><td align="left">红</td><td align="left">255</td><td align="left">0</td><td align="left">0</td></tr>
<tr><td align="left">绿</td><td align="left">0</td><td align="left">255</td><td align="left">0</td></tr>
<tr><td align="left">蓝</td><td align="left">0</td><td align="left">0</td><td align="left">255</td></tr>
<tr><td align="left">黄</td><td align="left">255</td><td align="left">255</td><td align="left">0</td></tr>
<tr><td align="left">白</td><td align="left">255</td><td align="left">255</td><td align="left">255</td></tr>
<tr><td align="left">黑</td><td align="left">0</td><td align="left">0</td><td align="left">0</td></tr>
</table>

## 本节小结

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">图像类型</th><th align="left">NumPy 数组形状</th><th align="left">每个像素</th></tr>
<tr><td align="left">灰度图</td><td align="left">(高, 宽)</td><td align="left">1 个数（0-255）</td></tr>
<tr><td align="left">彩色图</td><td align="left">(高, 宽, 3)</td><td align="left">3 个数（R, G, B）</td></tr>
</table>

理解了图像的数据本质，下一节我们学习用 OpenCV 对真实图片做各种操作。

---

## 本节练习

**练习 1（选择）**：一张 `640×480` 的彩色图，对应的 NumPy 数组形状是？
- A. `(640, 480)`
- B. `(480, 640)`
- C. `(480, 640, 3)`
- D. `(640, 480, 3)`

**练习 2（填空）**：在 8 位图像中，像素值 `0` 表示 ______，`255` 表示 ______。

**练习 3（代码）**：用 NumPy 构造一张 `50×50` 的灰度图，要求**上半部分全黑、下半部分全白**，并用 matplotlib 显示。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/01.02_opencv_basics/answers.txt


---

## 第二部分：OpenCV 基础操作——读写、灰度、绘图


In [ ]:
# ===== 中文字体配置（解决 matplotlib 中文乱码）=====
import matplotlib.pyplot as plt
import matplotlib
import platform

# 按系统选择合适的中文字体
_sys = platform.system()
if _sys == "Linux":
    # CANNLab 云环境通常有文泉驿或思源字体
    matplotlib.rcParams["font.sans-serif"] = ["WenQuanYi Zen Hei", "WenQuanYi Micro Hei",
                                              "Noto Sans CJK SC", "Source Han Sans SC",
                                              "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
elif _sys == "Darwin":
    matplotlib.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "Heiti TC"]
    matplotlib.rcParams["axes.unicode_minus"] = False
elif _sys == "Windows":
    matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
    matplotlib.rcParams["axes.unicode_minus"] = False
print(f"字体配置完成（系统: {_sys}）")
# ===== 中文字体配置结束 =====

import numpy as np
import cv2

# 构造一张 200x200 的测试图（灰色背景 + 彩色形状）
img = np.full((200, 200, 3), 200, dtype=np.uint8)

# 用 OpenCV 绘图函数画一些东西（注意 OpenCV 用 BGR 顺序！）
cv2.rectangle(img, (30, 30), (90, 90), (0, 0, 200), -1)      # 红色实心方块（B=0,G=0,R=200）
cv2.circle(img, (150, 60), 30, (0, 200, 0), -1)               # 绿色实心圆
cv2.line(img, (20, 150), (180, 150), (200, 0, 0), 3)          # 蓝色横线
cv2.putText(img, "Hello AI", (40, 185), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 100), 2)

# 保存到本地
cv2.imwrite("test_image.png", img)
print("✅ 测试图已保存为 test_image.png，形状:", img.shape)


### ⚠️ BGR vs RGB：OpenCV 初学者最大的坑

`cv2.imread()` 读取的图片默认是 **BGR 顺序**（蓝-绿-红），而 Matplotlib 的 `plt.imshow()` 期望 **RGB 顺序**（红-绿-蓝）。如果直接显示，**红色和蓝色会互换**。

**解决方法**：用 `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` 转换通道顺序。这是后面 YOLO 可视化时每次都要做的操作，务必记住。

In [ ]:
# ===== 错误示范：直接显示（颜色会反掉）=====
img_bgr = cv2.imread("test_image.png")   # OpenCV 默认读取为 BGR

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_bgr)   # 直接显示 BGR 图，红蓝反掉
plt.title("Wrong: BGR shown directly (R/B swapped)")
plt.axis('off')

# ===== 正确做法：BGR → RGB 转换后再显示 =====
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.subplot(1, 2, 2)
plt.imshow(img_rgb)
plt.title("Correct: Converted to RGB")
plt.axis('off')
plt.show()


## 1. 灰度转换

把彩色图转成灰度图是常见的预处理步骤（减少数据量、突出形状）。用 `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)`：

In [ ]:
# 彩色图转灰度图
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print("彩色图形状:", img_bgr.shape, "→ 灰度图形状:", img_gray.shape)
print("转换后每个像素是单个 0-255 的值，不再有 3 通道")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original (Color)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_gray, cmap='gray')
plt.title("Grayscale")
plt.axis('off')
plt.show()


## 2. OpenCV 绘图函数速查

绘图函数是后续"在 YOLO 检测结果上画框"的基础。OpenCV 的绘图都是**原地修改**（in-place），即直接在传入的图像数组上画，不会返回新图。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">函数</th><th align="left">作用</th><th align="left">关键参数</th></tr>
<tr><td align="left"><code>cv2.rectangle</code></td><td align="left">画矩形</td><td align="left"><code>img, (x1,y1), (x2,y2), color, thickness</code></td></tr>
<tr><td align="left"><code>cv2.circle</code></td><td align="left">画圆</td><td align="left"><code>img, center, radius, color, thickness</code></td></tr>
<tr><td align="left"><code>cv2.line</code></td><td align="left">画线</td><td align="left"><code>img, (x1,y1), (x2,y2), color, thickness</code></td></tr>
<tr><td align="left"><code>cv2.putText</code></td><td align="left">写文字</td><td align="left"><code>img, text, (x,y), font, size, color, thickness</code></td></tr>
</table>

> 💡 `thickness=-1` 表示**实心填充**；`thickness=正数` 表示线条粗细。

**坐标系约定**：OpenCV 中坐标是 `(x, y)` 即 `(列, 行)`，与 NumPy 索引 `[y, x]` 即 `[行, 列]` 相反，这也是常踩的坑。

我们用一个综合例子演示：在一张图上画出"模拟的 YOLO 检测框 + 标签"，这正是 02 章会用到的可视化技巧：

In [ ]:
# 综合演示：模拟 YOLO 检测结果可视化
# 假设 YOLO 检测到一个物体，边界框为 (40, 50) 到 (160, 170)，类别 "fruit"，置信度 0.92
demo = np.full((220, 220, 3), 240, dtype=np.uint8)   # 浅灰背景

# 1. 画边界框（绿色，粗细 3）
box_color = (0, 200, 0)   # BGR 中的绿色
cv2.rectangle(demo, (40, 50), (160, 170), box_color, 3)

# 2. 写标签文字（在框的左上角）
label = "fruit 0.92"
cv2.putText(demo, label, (40, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.6, box_color, 2)

# 3. 转成 RGB 后显示
plt.imshow(cv2.cvtColor(demo, cv2.COLOR_BGR2RGB))
plt.title("Simulated YOLO BBox (visualization for Ch.2)")
plt.axis('off')
plt.show()


这个"画框 + 写标签"的模式，就是 YOLO 检测结果可视化的核心。第 2 章我们会看到 YOLO 库已经封装好了 `result.plot()` 方法自动完成这件事。

---

## 本节练习

**练习 1（选择）**：为什么用 `cv2.imread()` 读取的图片直接用 `plt.imshow()` 显示会"红蓝反掉"？
- A. OpenCV 读取时压缩了图片
- B. OpenCV 用 BGR 顺序，Matplotlib 用 RGB 顺序
- C. Matplotlib 不支持彩色图
- D. OpenCV 读取的图片是倒着的

**练习 2（填空）**：OpenCV 绘图函数中，坐标格式是 `(_____, _____)` 即 `(列, 行)`，而 NumPy 数组索引是 `[_____, _____]` 即 `[行, 列]`。

**练习 3（代码）**：读取一张图片（或用 NumPy 构造），完成以下操作：
1. 在图片中央画一个红色的实心圆（半径 30）
2. 在圆下方写上文字 "Center"
3. 转成 RGB 后用 matplotlib 显示

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/01.02_opencv_basics/answers_opencv_ops.txt
